# Exercise MegaDetector on Local Images

This notebook runs MegaDetector on a local image folder:

- `/Users/elhorte/Pictures/project-id-tests`

It is safe to run even when the folder is currently empty.

## Step 1 — Define paths

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
md_root = repo_root / "third-party" / "MegaDetector"
image_dir = Path("/Users/elhorte/Pictures/project-id-tests")
output_dir = repo_root / "outputs" / "megadetector"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"MegaDetector path: {md_root}")
print(f"Image directory: {image_dir}")
print(f"Output directory: {output_dir}")

## Step 2 — Check image folder contents

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
images = sorted([p for p in image_dir.glob("**/*") if p.is_file() and p.suffix.lower() in image_exts])

print(f"Found {len(images)} image(s)")
for p in images[:20]:
    print(" -", p)

if not images:
    print("\nNo images found yet. Populate the folder and rerun this notebook.")

## Step 3 — Build an image list file for MegaDetector batch inference

In [ ]:
image_list_file = output_dir / "image_list.txt"
image_list_file.write_text("\n".join(str(p) for p in images), encoding="utf-8")
print(f"Wrote: {image_list_file}")

## Step 4 — Run MegaDetector (batch mode)

This uses the MegaDetector batch detection script and writes a JSON results file.

> If your local MegaDetector checkout uses a different entry script, update the command in the next cell accordingly.

In [6]:
import subprocess
import sys

results_json = output_dir / "megadetector_results.json"
batch_script = md_root / "megadetector" / "detection" / "run_detector_batch.py"

if not md_root.exists():
    raise FileNotFoundError(f"MegaDetector repo not found: {md_root}")
if not batch_script.exists():
    raise FileNotFoundError(f"Batch script not found: {batch_script}")
if len(images) == 0:
    raise RuntimeError("No images available for inference. Add images and rerun.")

cmd = [
    sys.executable,
    str(batch_script),
    "MDV5A",
    str(image_list_file),
    str(results_json),
    "--recursive",
]

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(f"\nDone. Results saved to: {results_json}")

Running command:
/Users/elhorte/git/earth-biometrics/project-id/.venv/bin/python /Users/elhorte/git/earth-biometrics/project-id/notebooks/third-party/MegaDetector/megadetector/detection/run_detector_batch.py MDV5A /Users/elhorte/git/earth-biometrics/project-id/notebooks/outputs/megadetector/image_list.txt /Users/elhorte/git/earth-biometrics/project-id/notebooks/outputs/megadetector/megadetector_results.json --recursive
Model v5a.0.1 already exists and is valid at /var/folders/5s/t45jh9hx2z9ccj91wyt3xvjm0000gn/T/megadetector_models/md_v5a.0.1.pt
Loaded 62 image filenames from .txt list file /Users/elhorte/git/earth-biometrics/project-id/notebooks/outputs/megadetector/image_list.txt
PyTorch reports 0 available CUDA devices (load_and_run_detector_batch)
PyTorch reports Metal Performance Shaders are available
PyTorch reports 0 available CUDA devices (load_detector)
PyTorch reports Metal Performance Shaders are available


/Users/elhorte/git/earth-biometrics/project-id/.venv/lib/python3.13/site-packages/yolov5/utils/general.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Loading PT detector with compatibility mode classic
Loaded image size 1280 from model metadata
Using model stride: 64
Using MPS device
PTDetector using device mps


Fusing layers... 
Fusing layers... 
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs
Model summary: 733 layers, 140054656 parameters, 0 gradients, 208.8 GFLOPs


Loaded model in 2.4 seconds


100%|██████████| 62/62 [00:06<00:00,  9.03it/s]


Finished inference for 62 images in 9.83 seconds (6.30 images per second)
Output file saved at /Users/elhorte/git/earth-biometrics/project-id/notebooks/outputs/megadetector/megadetector_results.json
Done, thanks for MegaDetect'ing!

Done. Results saved to: /Users/elhorte/git/earth-biometrics/project-id/notebooks/outputs/megadetector/megadetector_results.json


## Step 5 — Preview detections summary

In [ ]:
import json
from collections import Counter

results_json = output_dir / "megadetector_results.json"
data = json.loads(results_json.read_text(encoding="utf-8"))
images_data = data.get("images", [])

print(f"Images in results: {len(images_data)}")

detections_per_image = Counter()
for item in images_data:
    detections_per_image[len(item.get("detections", []))] += 1

print("Detection count distribution (detections -> number of images):")
for k in sorted(detections_per_image):
    print(f"  {k} -> {detections_per_image[k]}")

## Step 6 — Visualize detections (bounding-box QA)

Draws MegaDetector bounding boxes on a few sample images, saves annotated copies to
`outputs/megadetector/annotated/`, and displays them inline for quick quality checking.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display
import megadetector.visualization.visualization_utils as vis_utils

# Reuse output_dir from Step 1; results file from Step 4.
results_json = output_dir / "megadetector_results.json"
results = json.loads(results_json.read_text(encoding="utf-8"))
category_map = results.get("detection_categories", {})

confidence_threshold = 0.2   # only draw boxes at/above this confidence
max_samples = 6              # how many annotated images to preview

annotated_dir = output_dir / "annotated"
annotated_dir.mkdir(parents=True, exist_ok=True)

images_with_detections = [
    im for im in results.get("images", [])
    if not im.get("failure")
    and any(d.get("conf", 0) >= confidence_threshold for d in im.get("detections", []))
]

print(f"{len(images_with_detections)} image(s) have detections at conf >= {confidence_threshold}")
if not images_with_detections:
    print("Nothing to visualize yet. Run Step 4 on a folder that contains animals/people/vehicles.")

for im in images_with_detections[:max_samples]:
    src = Path(im["file"])
    if not src.exists():
        print(f"  (skipped, missing file) {src}")
        continue

    image = vis_utils.load_image(str(src))
    vis_utils.render_detection_bounding_boxes(
        im["detections"], image,
        label_map=category_map,
        confidence_threshold=confidence_threshold,
        thickness=4,
    )

    out_path = annotated_dir / f"{src.stem}_annotated.png"
    image.save(out_path)

    n_boxes = sum(1 for d in im["detections"] if d.get("conf", 0) >= confidence_threshold)
    print(f"  {src.name}: {n_boxes} box(es) -> {out_path.name}")
    display(IPyImage(filename=str(out_path), width=700))

print(f"\nAnnotated images saved to: {annotated_dir}")